In [ ]:
# -*- coding: utf-8 -*-
"""NDVI minimum composite at original 10m x 10m grid size using tiling + auto token renewal"""

!pip install rasterio oauthlib requests requests-oauthlib pyproj

from oauthlib.oauth2 import BackendApplicationClient
from requests_oauthlib import OAuth2Session
import numpy as np
import tarfile
import io
import os
import time
import rasterio
from rasterio.io import MemoryFile
from rasterio.merge import merge
from google.colab import drive
from pyproj import Transformer

# -----------------------------
# 1. Mount Google Drive
# -----------------------------
drive.mount('/content/drive')

output_dir = '/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/ndvi_10m_tiled'
tile_dir = os.path.join(output_dir, "tiles")
os.makedirs(output_dir, exist_ok=True)
os.makedirs(tile_dir, exist_ok=True)

# -----------------------------
# 2. Copernicus credentials
# -----------------------------
client_id = "YOUR_COPERNICUS_CLIENT_ID"
client_secret = "YOUR_COPERNICUS_CLIENT_SECRET"

# -----------------------------
# 3. Authentication
# -----------------------------
def getauth_token():
    client = BackendApplicationClient(client_id=client_id)
    oauth = OAuth2Session(client=client)
    token = oauth.fetch_token(
        token_url="https://identity.cloudferro.com/auth/realms/CDSE/protocol/openid-connect/token",
        client_id=client_id,
        client_secret=client_secret,
    )
    return oauth, token

def ensure_valid_oauth(oauth, token, buffer_seconds=120):
    """
    Re-authenticate if token is missing or will expire soon.
    buffer_seconds=120 -> renew 2 minutes before expiry
    """
    now = time.time()
    expires_at = token.get("expires_at", None)

    if expires_at is None or now >= (expires_at - buffer_seconds):
        print("Token expired or about to expire. Fetching new token...")
        oauth, token = getauth_token()

    return oauth, token

# -----------------------------
# 4. Time slots
# -----------------------------
slots = []
for month in ['01', '02', '03', '04', '10', '11', '12']:
    month_group = [
        (f"2021-{month}-01", f"2021-{month}-28"),
        (f"2022-{month}-01", f"2022-{month}-28"),
        (f"2023-{month}-01", f"2023-{month}-28"),
        (f"2024-{month}-01", f"2024-{month}-28")
    ]
    slots.append(month_group)

# -----------------------------
# 5. Evalscript
# -----------------------------
evalscript = """
function setup() {
  return {
    input: ["B04", "B08", "dataMask"],
    output: {
      id: "default",
      bands: 3,
      sampleType: "FLOAT32"
    }
  };
}

function evaluatePixel(sample) {
  return [sample.B04, sample.B08, sample.dataMask];
}
"""

# -----------------------------
# 6. AOI bbox in EPSG:4326
# -----------------------------
lon_min, lat_min = 81.45785785, 29.06047142
lon_max, lat_max = 82.04838275, 29.40491605

# Convert to UTM 44N so resx/resy=10 really means 10 meters
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32644", always_xy=True)
x_min, y_min = transformer.transform(lon_min, lat_min)
x_max, y_max = transformer.transform(lon_max, lat_max)

# -----------------------------
# 7. Create tiles
# -----------------------------
# Process API max is 2500 px in width/height.
# At 10 m, 20000 m tile = 2000 px, safely within limit.
tile_size_m = 20000

def create_tiles(xmin, ymin, xmax, ymax, tile_size):
    tiles = []
    x = xmin
    while x < xmax:
        x2 = min(x + tile_size, xmax)
        y = ymin
        while y < ymax:
            y2 = min(y + tile_size, ymax)
            tiles.append([x, y, x2, y2])
            y = y2
        x = x2
    return tiles

tiles = create_tiles(x_min, y_min, x_max, y_max, tile_size_m)
print(f"Number of tiles: {len(tiles)}")

# -----------------------------
# 8. Tile request
# -----------------------------
def get_tile_request(slot, bbox_utm, oauth, token):
    oauth, token = ensure_valid_oauth(oauth, token)

    request = {
        "input": {
            "bounds": {
                "bbox": bbox_utm,
                "properties": {
                    "crs": "http://www.opengis.net/def/crs/EPSG/0/32644"
                }
            },
            "data": [{
                "type": "sentinel-2-l2a",
                "dataFilter": {
                    "timeRange": {
                        "from": slot[0] + "T00:00:00Z",
                        "to": slot[1] + "T23:59:59Z"
                    },
                    "maxCloudCoverage": 15
                }
            }]
        },
        "output": {
            "resx": 10,
            "resy": 10,
            "responses": [{
                "identifier": "default",
                "format": {
                    "type": "image/tiff"
                }
            }]
        },
        "evalscript": evalscript
    }

    url = "https://sh.dataspace.copernicus.eu/api/v1/process"
    response = oauth.post(url, json=request, headers={"Accept": "application/tar"})

    if response.status_code != 200:
        raise RuntimeError(f"Error {response.status_code}: {response.text}")

    return response, oauth, token

# -----------------------------
# 9. NDVI calculation
# -----------------------------
def calculate_ndvi(red, nir, mask):
    red = red.astype(np.float32)
    nir = nir.astype(np.float32)
    mask = mask.astype(np.float32)

    ndvi = np.full(red.shape, np.nan, dtype=np.float32)
    valid = (mask == 1) & ((nir + red) != 0)
    ndvi[valid] = (nir[valid] - red[valid]) / (nir[valid] + red[valid])

    return ndvi

# -----------------------------
# 10. Safe pixel-wise minimum
# -----------------------------
def nan_safe_min(arr1, arr2):
    if arr1 is None:
        return arr2.copy()
    if arr2 is None:
        return arr1.copy()

    return np.where(
        np.isnan(arr1), arr2,
        np.where(np.isnan(arr2), arr1, np.minimum(arr1, arr2))
    )

# -----------------------------
# 11. Process one date: download tiles, compute NDVI, merge
# -----------------------------
def process_one_date(slot, oauth, token, tiles, tile_dir, output_dir):
    ndvi_tile_paths = []

    for idx, tile_bbox in enumerate(tiles):
        print(f"  Tile {idx + 1}/{len(tiles)}")

        resp, oauth, token = get_tile_request(slot, tile_bbox, oauth, token)
        tar_bytes = resp.content
        tar_stream = io.BytesIO(tar_bytes)

        with tarfile.open(fileobj=tar_stream, mode='r:*') as tar:
            tif_member = next((m for m in tar.getmembers() if m.name.endswith('.tif')), None)
            if tif_member is None:
                raise ValueError("No .tif file found in the archive")

            tif_file = tar.extractfile(tif_member)
            tif_bytes = tif_file.read()

            with MemoryFile(tif_bytes) as memfile:
                with memfile.open() as dataset:
                    red = dataset.read(1).astype(np.float32)
                    nir = dataset.read(2).astype(np.float32)
                    data_mask = dataset.read(3).astype(np.float32)

                    ndvi = calculate_ndvi(red, nir, data_mask)

                    profile = dataset.profile.copy()
                    profile.update(
                        dtype=rasterio.float32,
                        count=1,
                        nodata=np.nan
                    )

                    tile_path = os.path.join(tile_dir, f"ndvi_{slot[0]}_tile_{idx}.tif")
                    with rasterio.open(tile_path, "w", **profile) as dst:
                        dst.write(ndvi, 1)

                    ndvi_tile_paths.append(tile_path)

    # Merge tiles
    src_files = [rasterio.open(p) for p in ndvi_tile_paths]
    mosaic, out_transform = merge(src_files)

    out_meta = src_files[0].meta.copy()
    out_meta.update({
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": out_transform,
        "count": 1,
        "dtype": "float32",
        "nodata": np.nan
    })

    merged_path = os.path.join(output_dir, f"ndvi_{slot[0]}.tif")
    with rasterio.open(merged_path, "w", **out_meta) as dest:
        dest.write(mosaic[0].astype(np.float32), 1)

    for src in src_files:
        src.close()

    # Optional: delete temporary tile files after merging
    for p in ndvi_tile_paths:
        if os.path.exists(p):
            os.remove(p)

    return merged_path, out_meta, oauth, token

# -----------------------------
# 12. Main processing
# -----------------------------
oauth, token = getauth_token()

absolute_min_ndvi = None
absolute_profile = None

for i, month_group in enumerate(slots):
    month_min_ndvi = None
    month_profile = None

    for slot in month_group:
        print(f"Processing: {slot[0]} to {slot[1]}")

        merged_ndvi_path, profile, oauth, token = process_one_date(
            slot, oauth, token, tiles, tile_dir, output_dir
        )

        with rasterio.open(merged_ndvi_path) as src:
            ndvi = src.read(1).astype(np.float32)

        month_min_ndvi = nan_safe_min(month_min_ndvi, ndvi)
        month_profile = profile

    month_name = month_group[0][0][5:7]
    month_min_path = os.path.join(output_dir, f"min_ndvi_month_{month_name}.tif")
    with rasterio.open(month_min_path, "w", **month_profile) as dst:
        dst.write(month_min_ndvi.astype(np.float32), 1)

    print(f"Saved month minimum: {month_min_path}")

    absolute_min_ndvi = nan_safe_min(absolute_min_ndvi, month_min_ndvi)
    absolute_profile = month_profile

# -----------------------------
# 13. Save final absolute minimum
# -----------------------------
final_path = os.path.join(output_dir, "absolute_min_ndvi_10m.tif")
with rasterio.open(final_path, "w", **absolute_profile) as dst:
    dst.write(absolute_min_ndvi.astype(np.float32), 1)

print("Done.")
print(f"Final absolute minimum NDVI saved at: {final_path}")

Mounted at /content/drive
Number of tiles: 6
Processing: 2021-01-01 to 2021-01-28
  Tile 1/6


  Tile 2/6


  Tile 3/6


  Tile 4/6


  Tile 5/6


  Tile 6/6


... (159 outputs left out)

  Tile 4/6


  Tile 5/6


  Tile 6/6


Saved month minimum: /content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/ndvi_10m_tiled/min_ndvi_month_12.tif
Done.
Final absolute minimum NDVI saved at: /content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/ndvi_10m_tiled/absolute_min_ndvi_10m.tif
